# Validate TD and FD→TD waveform-mode paths

This notebook compares `NRSur7dq4` TD modes with `IMRPhenomXPHM` FD→TD modes. Shared physical/numerical parameters are identical except for the requested starting-frequency convention:

- `NRSur7dq4`: `f_lower = 0.0`;
- `IMRPhenomXPHM`: `f_lower = 10.0`.

`phi_ref` is now kept the same for both approximants. The notebook measures the residual constant complex phase offset between the generated `h22` modes in a chosen pre-peak alignment window, prints it, and plots phase-coaligned waveforms.

All plotted time axes are geometric: `(t - t_peak)/M`. Plot helpers accept either `xlim=[...]` or `xlim_m=[...]`; both are interpreted in units of total mass `M`, not seconds.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import lal

from waveformtools.models.lal import LALWaveformModel
from waveformtools.modes_array import ModesArray
from lalsimulation import SimInspiralGetApproximantFromString

try:
    from pycbc.waveform.utils import coalign_waveforms
except Exception:
    coalign_waveforms = None


In [ ]:
TD_APPROXIMANT = "NRSur7dq4"
FD_APPROXIMANT = "IMRPhenomXPHM"
TD_F_LOWER = 0.0
FD_F_LOWER = 10.0
MODE_TO_PLOT = (2, 2)

# Same phase reference for both approximants. Any residual phase difference
# is measured from the generated modes, not inserted by hand.
PHI_REF = 0.3

COMMON_PARAMS = dict(
    mass1=35.0, mass2=30.0,
    spin1x=0.0, spin1y=0.0, spin1z=0.2,
    spin2x=0.0, spin2y=0.0, spin2z=-0.1,
    distance=400.0, inclination=0.7, phi_ref=PHI_REF,
    f_ref=20.0, f_max=512.0,
    delta_t=1.0 / 4096.0, delta_f=1.0 / 8.0, ell_max=4,
)

M_SECONDS = (COMMON_PARAMS["mass1"] + COMMON_PARAMS["mass2"]) * lal.MTSUN_SI
print(f"Total-mass time unit: M = {M_SECONDS:.9e} s")
print(f"Common phi_ref = {PHI_REF}")
print(f"PyCBC coalign_waveforms available: {coalign_waveforms is not None}")

# Geometric-time plot/alignment windows in units of total mass M.
PLOT_XLIM_FULL = None
PLOT_XLIM_NEAR_PEAK_M = [-500.0, 100.0]
PLOT_XLIM_PREPEAK_M = [-500.0, 0.0]
ALIGN_XLIM_M = [-300.0, -50.0]


In [ ]:
def make_params(approximant):
    params = dict(COMMON_PARAMS)
    params["approximant"] = approximant
    if approximant == TD_APPROXIMANT:
        params["f_lower"] = TD_F_LOWER
    elif approximant == FD_APPROXIMANT:
        params["f_lower"] = FD_F_LOWER
    else:
        raise ValueError(f"Unexpected approximant {approximant!r}")
    return params


def lal_string_is_available(approximant):
    try:
        SimInspiralGetApproximantFromString(approximant)
        return True
    except Exception:
        return False


def print_param_difference(td_params, fd_params):
    keys = sorted(set(td_params) | set(fd_params))
    print("Parameter comparison: TD vs FD")
    for key in keys:
        td_val = td_params.get(key)
        fd_val = fd_params.get(key)
        flag = "DIFF" if td_val != fd_val else "same"
        print(f"{key:>16s}: {td_val!r:>20}   {fd_val!r:>20}   {flag}")


def geometric_peak_time(wfm, mode=MODE_TO_PLOT, m_seconds=M_SECONDS):
    y = np.asarray(wfm.mode(*mode))
    t_raw = np.asarray(wfm.time_axis)
    if len(t_raw) != len(y):
        return np.arange(len(y), dtype=float)
    i_peak = int(np.argmax(np.abs(y)))
    return (t_raw - t_raw[i_peak]) / m_seconds


def normalized_mode(wfm, mode=MODE_TO_PLOT, phase_shift=0.0):
    y = np.asarray(wfm.mode(*mode)) * np.exp(1j * phase_shift)
    scale = np.max(np.abs(y))
    if scale == 0 or not np.isfinite(scale):
        scale = 1.0
    return y / scale


def _resolve_xlim(xlim=None, xlim_m=None):
    if xlim is not None and xlim_m is not None:
        raise ValueError("Pass only one of xlim or xlim_m")
    return xlim if xlim is not None else xlim_m


In [ ]:
def summarize_modes(wfm, name):
    print(f"\n{name}")
    print("-" * len(name))
    print("type:", type(wfm))
    print("label:", getattr(wfm, "label", None))
    print("ell_max:", getattr(wfm, "ell_max", None))
    print("data_len:", getattr(wfm, "data_len", None))
    try:
        print("raw time range:", float(wfm.time_axis[0]), float(wfm.time_axis[-1]), "N=", len(wfm.time_axis))
        gt = geometric_peak_time(wfm)
        print("geometric peak-centered range [M]:", float(gt[0]), float(gt[-1]))
    except Exception as exc:
        print("time unavailable:", repr(exc))


def relative_l2(a, b, eps=1e-300):
    a = np.asarray(a)
    b = np.asarray(b)
    return np.linalg.norm(a - b) / max(np.linalg.norm(a), np.linalg.norm(b), eps)


def compare_common_modes(a, b, label_a="a", label_b="b", rtol=1e-10, atol=1e-12):
    failures = []
    n_common = 0
    ell_max = min(int(a.ell_max), int(b.ell_max))
    for ell in range(2, ell_max + 1):
        for m in range(-ell, ell + 1):
            try:
                am = np.asarray(a.mode(ell, m))
                bm = np.asarray(b.mode(ell, m))
            except Exception:
                continue
            if am.shape != bm.shape:
                failures.append((ell, m, "shape_mismatch", am.shape, bm.shape))
                continue
            n_common += 1
            if not np.allclose(am, bm, rtol=rtol, atol=atol):
                failures.append((ell, m, float(np.max(np.abs(am - bm))), float(relative_l2(am, bm))))
    print(f"Compared {n_common} common modes: {label_a} vs {label_b}")
    print(f"Failures / mismatches: {len(failures)}")
    for item in failures[:20]:
        print(item)
    return failures


In [ ]:
def interpolate_complex(t_src, h_src, t_dst):
    order = np.argsort(t_src)
    t_src = np.asarray(t_src)[order]
    h_src = np.asarray(h_src)[order]
    real = np.interp(t_dst, t_src, h_src.real)
    imag = np.interp(t_dst, t_src, h_src.imag)
    return real + 1j * imag


def best_constant_mode_phase_shift(ref_wfm, test_wfm, mode=MODE_TO_PLOT, xlim_m=ALIGN_XLIM_M):
    """Return alpha such that test * exp(i alpha) best aligns to ref."""
    t_ref = geometric_peak_time(ref_wfm, mode=mode)
    t_test = geometric_peak_time(test_wfm, mode=mode)
    h_ref = np.asarray(ref_wfm.mode(*mode))
    h_test = np.asarray(test_wfm.mode(*mode))

    lo, hi = xlim_m
    common_lo = max(lo, float(np.min(t_ref)), float(np.min(t_test)))
    common_hi = min(hi, float(np.max(t_ref)), float(np.max(t_test)))
    mask = (t_ref >= common_lo) & (t_ref <= common_hi)
    if np.count_nonzero(mask) < 8:
        raise RuntimeError("Too few samples in the phase-alignment window")

    tref = t_ref[mask]
    href = h_ref[mask]
    htest = interpolate_complex(t_test, h_test, tref)

    # Least-squares constant complex phase: minimize ||href - exp(i alpha) htest||.
    alpha = np.angle(np.vdot(htest, href))
    before = relative_l2(href, htest)
    after = relative_l2(href, htest * np.exp(1j * alpha))
    return alpha, before, after, (common_lo, common_hi)


def equivalent_orbital_phase_shift(mode_phase_shift, m=MODE_TO_PLOT[1]):
    # If h_lm -> exp(-i m phi_ref) h_lm, then a mode rotation alpha
    # corresponds to delta phi_ref = -alpha/m. Check the sign convention by plotting.
    return -mode_phase_shift / float(m)


In [ ]:
def _unpack_waveform_item(item):
    if len(item) == 2:
        label, wfm = item
        phase_shift = 0.0
    elif len(item) == 3:
        label, wfm, phase_shift = item
    else:
        raise ValueError("waveform entries must be (label, wfm) or (label, wfm, phase_shift)")
    return label, wfm, phase_shift


def plot_mode_real(waveforms, title, mode=MODE_TO_PLOT, normalize=False, xlim=None, xlim_m=None):
    xlim_use = _resolve_xlim(xlim=xlim, xlim_m=xlim_m)
    plt.figure(figsize=(8, 3.5))
    for item in waveforms:
        label, wfm, phase_shift = _unpack_waveform_item(item)
        y = normalized_mode(wfm, mode=mode, phase_shift=phase_shift) if normalize else np.asarray(wfm.mode(*mode)) * np.exp(1j * phase_shift)
        x = geometric_peak_time(wfm, mode=mode)
        plt.plot(x, y.real, label=label)
    if xlim_use is not None:
        plt.xlim(*xlim_use)
    plt.xlabel(r"$(t - t_{\rm peak})/M$")
    plt.ylabel(f"Re h_{{{mode[0]}{mode[1]}}}" + (" / max|h|" if normalize else ""))
    plt.title(title)
    plt.legend()
    plt.tight_layout()


def plot_mode_amp(waveforms, title, mode=MODE_TO_PLOT, normalize=True, xlim=None, xlim_m=None):
    xlim_use = _resolve_xlim(xlim=xlim, xlim_m=xlim_m)
    plt.figure(figsize=(8, 3.5))
    for item in waveforms:
        label, wfm, phase_shift = _unpack_waveform_item(item)
        y = normalized_mode(wfm, mode=mode, phase_shift=phase_shift) if normalize else np.asarray(wfm.mode(*mode)) * np.exp(1j * phase_shift)
        x = geometric_peak_time(wfm, mode=mode)
        plt.plot(x, np.abs(y), label=label)
    if xlim_use is not None:
        plt.xlim(*xlim_use)
    plt.xlabel(r"$(t - t_{\rm peak})/M$")
    plt.ylabel(f"|h_{{{mode[0]}{mode[1]}}}|" + (" / max|h|" if normalize else ""))
    plt.title(title)
    plt.legend()
    plt.tight_layout()


def plot_route_difference(reference, other, ref_label, other_label, mode=MODE_TO_PLOT, xlim=None, xlim_m=None):
    xlim_use = _resolve_xlim(xlim=xlim, xlim_m=xlim_m)
    ref = np.asarray(reference.mode(*mode))
    oth = np.asarray(other.mode(*mode))
    n = min(len(ref), len(oth))
    diff = oth[:n] - ref[:n]
    t = geometric_peak_time(reference, mode=mode)[:n]
    plt.figure(figsize=(8, 3.2))
    plt.plot(t, diff.real, label="Re difference")
    plt.plot(t, diff.imag, label="Im difference")
    if xlim_use is not None:
        plt.xlim(*xlim_use)
    plt.xlabel(r"$(t - t_{\rm peak})/M$")
    plt.ylabel(f"{other_label} - {ref_label}")
    plt.title(f"FD→TD route difference for h_{{{mode[0]}{mode[1]}}}")
    plt.legend()
    plt.tight_layout()


## Generate and verify waveforms

In [ ]:
td_params = make_params(TD_APPROXIMANT)
fd_params = make_params(FD_APPROXIMANT)
print_param_difference(td_params, fd_params)

expected_differences = {"approximant", "f_lower"}
actual_differences = {k for k in sorted(set(td_params) | set(fd_params)) if td_params.get(k) != fd_params.get(k)}
assert actual_differences == expected_differences, actual_differences

assert lal_string_is_available(TD_APPROXIMANT), f"{TD_APPROXIMANT} not available in this LALSuite build"
assert lal_string_is_available(FD_APPROXIMANT), f"{FD_APPROXIMANT} not available in this LALSuite build"

td_model = LALWaveformModel(parameters_dict=td_params)
w_td_native = td_model.get_td_waveform_modes(dimensionless=False)
assert isinstance(w_td_native, ModesArray)
summarize_modes(w_td_native, f"native TD modes: {TD_APPROXIMANT}, f_lower={TD_F_LOWER}, phi_ref={PHI_REF}")

fd_model = LALWaveformModel(parameters_dict=fd_params)
w_fd = fd_model.get_fd_waveform_modes(dimensionless=False)
w_fd_as_td = fd_model.get_fd_waveform_modes_as_td(dimensionless=False)
w_fd_as_td_alias = fd_model.get_fd_modes_as_td(dimensionless=False)
w_fd_auto_td = fd_model.get_td_waveform_modes(dimensionless=False)
w_fd_manual_as_td = w_fd.to_time_basis()

for w in [w_fd, w_fd_as_td, w_fd_as_td_alias, w_fd_auto_td, w_fd_manual_as_td]:
    assert isinstance(w, ModesArray)
summarize_modes(w_fd, f"native FD modes: {FD_APPROXIMANT}, f_lower={FD_F_LOWER}, phi_ref={PHI_REF}")
summarize_modes(w_fd_as_td, f"FD modes as TD: {FD_APPROXIMANT}, f_lower={FD_F_LOWER}, phi_ref={PHI_REF}")


## Compare FD→TD routes

In [ ]:
fd_td_routes = [
    ("explicit get_fd_waveform_modes_as_td", w_fd_as_td),
    ("alias get_fd_modes_as_td", w_fd_as_td_alias),
    ("automatic get_td_waveform_modes", w_fd_auto_td),
    ("manual FD.to_time_basis", w_fd_manual_as_td),
]

assert len(compare_common_modes(w_fd_as_td, w_fd_as_td_alias, "explicit", "alias", rtol=0, atol=0)) == 0
assert len(compare_common_modes(w_fd_as_td, w_fd_auto_td, "explicit", "automatic", rtol=0, atol=0)) == 0
assert len(compare_common_modes(w_fd_as_td, w_fd_manual_as_td, "explicit", "manual", rtol=0, atol=0)) == 0

w_fd_auto_td_dimless = fd_model.get_td_waveform_modes(dimensionless=True)
w_fd_as_td_dimless = fd_model.get_fd_waveform_modes_as_td(dimensionless=True)
assert len(compare_common_modes(w_fd_auto_td_dimless, w_fd_as_td_dimless, "dimensionless automatic", "dimensionless explicit", rtol=0, atol=0)) == 0


## Measure constant h22 phase offset and coalign

In [ ]:
FD_H22_PHASE_SHIFT, phase_err_before, phase_err_after, used_window = best_constant_mode_phase_shift(
    w_td_native, w_fd_as_td, mode=MODE_TO_PLOT, xlim_m=ALIGN_XLIM_M
)
approx_orbital_shift = equivalent_orbital_phase_shift(FD_H22_PHASE_SHIFT, m=MODE_TO_PLOT[1])

print(f"Alignment window [M]: {used_window}")
print(f"Best constant mode phase shift alpha for FD h22: {FD_H22_PHASE_SHIFT:.12g} rad")
print(f"Equivalent orbital-phase shift, assuming h_lm -> exp(-i m phi) h_lm: {approx_orbital_shift:.12g} rad")
print(f"Relative L2 error before phase coalignment: {phase_err_before:.6e}")
print(f"Relative L2 error after  phase coalignment: {phase_err_after:.6e}")


## Visual comparisons

The `xlim` arguments below are lists in geometric units `M`.

In [ ]:
plot_mode_real(
    [(f"{TD_APPROXIMANT}", w_td_native),
     (f"{FD_APPROXIMANT} FD→TD, unaligned", w_fd_as_td)],
    title="Unaligned TD approximant vs FD→TD approximant: Re h22",
    normalize=True,
    xlim=PLOT_XLIM_PREPEAK_M,
)
plot_mode_real(
    [(f"{TD_APPROXIMANT}", w_td_native),
     (f"{FD_APPROXIMANT} FD→TD, phase-coaligned", w_fd_as_td, FD_H22_PHASE_SHIFT)],
    title="Phase-coaligned TD approximant vs FD→TD approximant: Re h22",
    normalize=True,
    xlim=PLOT_XLIM_PREPEAK_M,
)
plot_mode_amp(
    [(f"{TD_APPROXIMANT}", w_td_native),
     (f"{FD_APPROXIMANT} FD→TD", w_fd_as_td)],
    title="TD approximant vs FD→TD approximant: |h22|",
    normalize=True,
    xlim=PLOT_XLIM_PREPEAK_M,
)


## FD→TD route overlays and diagnostics

In [ ]:
plot_mode_real(fd_td_routes, title=f"FD→TD route comparison: {FD_APPROXIMANT}", normalize=False, xlim=PLOT_XLIM_FULL)
plot_mode_amp(fd_td_routes, title=f"FD→TD route amplitude comparison: {FD_APPROXIMANT}", normalize=True, xlim=PLOT_XLIM_FULL)
plot_mode_real(fd_td_routes, title=f"FD→TD route comparison near peak: {FD_APPROXIMANT}", normalize=False, xlim=PLOT_XLIM_NEAR_PEAK_M)
plot_mode_amp(fd_td_routes, title=f"FD→TD route amplitude near peak: {FD_APPROXIMANT}", normalize=True, xlim=PLOT_XLIM_NEAR_PEAK_M)

plot_route_difference(w_fd_as_td, w_fd_as_td_alias, "explicit", "alias", xlim=PLOT_XLIM_FULL)
plot_route_difference(w_fd_as_td, w_fd_auto_td, "explicit", "automatic", xlim=PLOT_XLIM_FULL)
plot_route_difference(w_fd_as_td, w_fd_manual_as_td, "explicit", "manual", xlim=PLOT_XLIM_FULL)

plot_mode_real([(f"{FD_APPROXIMANT} explicit FD→TD", w_fd_as_td)], "Full cyclic FD→TD output", normalize=True, xlim=PLOT_XLIM_FULL)
plot_mode_real(fd_td_routes, title=f"Cropped FD→TD route comparison: {FD_APPROXIMANT}", normalize=False, xlim=PLOT_XLIM_PREPEAK_M)


The post-peak low-amplitude oscillatory part in the full FD→TD output is usually a cyclic-IFFT/wraparound artifact or taper/finite-bandwidth residue, not a reliable physical late-time waveform segment. Use cropped pre-peak windows for visual comparisons, and do not blindly integrate the full cyclic FD→TD array for balance laws unless support, padding, tapering, and time-origin conventions are verified.